# Disease Prediction System

Updated to use the Kaggle **"Disease Prediction Using Machine Learning"** dataset (132 symptoms, 41 diseases) instead of the original 10-symptom dataset, which had 77% of rows sharing identical symptom patterns across different diseases (a hard accuracy ceiling).

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


## Load data

Using `Training.csv` for train/test split and evaluation, and `Testing.csv` as a fully independent holdout set to sanity-check generalization.

In [ ]:
df = pd.read_csv("Training.csv")

# Drop the stray empty trailing column present in this Kaggle file
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])

df.head()


In [ ]:
df.isnull().sum().sum()  # total missing values

In [ ]:
df.shape

In [ ]:
df['prognosis'].nunique(), df['prognosis'].value_counts()


## Encode target labels

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['prognosis'] = le.fit_transform(df['prognosis'])
df.head()


In [ ]:
X = df.drop("prognosis", axis=1)
Y = df["prognosis"]


## Check for the symptom-ambiguity issue from the old dataset

With 132 features instead of 10, this should be far less of a problem — worth confirming.

In [ ]:
grp = df.groupby(list(X.columns))['prognosis'].nunique()
print("unique symptom combos:", len(grp))
print("combos mapping to >1 disease:", (grp > 1).sum())


## Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)


## Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, Y_train)


In [ ]:
Y_pred = model.predict(X_test)

## KNN (with scaling)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, Y_train)


In [ ]:
Y_pred_knn = knn.predict(X_test_scaled)

## Compare Gradient Boosting vs KNN

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_gb = accuracy_score(Y_test, Y_pred)
precision_gb = precision_score(Y_test, Y_pred, average='weighted')
recall_gb = recall_score(Y_test, Y_pred, average='weighted')
f1_gb = f1_score(Y_test, Y_pred, average='weighted')

accuracy_knn = accuracy_score(Y_test, Y_pred_knn)
precision_knn = precision_score(Y_test, Y_pred_knn, average='weighted')
recall_knn = recall_score(Y_test, Y_pred_knn, average='weighted')
f1_knn = f1_score(Y_test, Y_pred_knn, average='weighted')

results = pd.DataFrame({
    "Model": ["Gradient Boosting", "KNN"],
    "Accuracy": [accuracy_gb, accuracy_knn],
    "Precision": [precision_gb, precision_knn],
    "Recall": [recall_gb, recall_knn],
    "F1 Score": [f1_gb, f1_knn]
})

print(results)


## Compare all 8 models

In [ ]:
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
    ExtraTreesClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

models = {
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "SVM": SVC(
        kernel='rbf',
        random_state=42
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Naive Bayes": GaussianNB()
}


In [ ]:
results = []

for name, m in models.items():

    m.fit(X_train, Y_train)

    pred = m.predict(X_test)

    accuracy = accuracy_score(Y_test, pred)
    precision = precision_score(
        Y_test, pred, average='weighted', zero_division=0
    )
    recall = recall_score(
        Y_test, pred, average='weighted', zero_division=0
    )
    f1 = f1_score(
        Y_test, pred, average='weighted', zero_division=0
    )

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

results_df


## Cross-validation on the best model

More reliable than a single split, and useful evidence for your report.

In [ ]:
from sklearn.model_selection import cross_val_score

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

cv_scores = cross_val_score(best_model, X, Y, cv=5)
print(f"{best_model_name} cross-val accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")


## Sanity check on the independent Testing.csv holdout

In [ ]:
holdout = pd.read_csv("Testing.csv")
holdout = holdout.drop(columns=[c for c in holdout.columns if c.startswith("Unnamed")])

X_holdout = holdout.drop("prognosis", axis=1)[X.columns]  # keep column order aligned
Y_holdout = le.transform(holdout["prognosis"])

best_model.fit(X, Y)  # refit best model on all of Training.csv
holdout_pred = best_model.predict(X_holdout)
print("Holdout accuracy:", accuracy_score(Y_holdout, holdout_pred))


## Save the best model

In [ ]:
import joblib

joblib.dump(best_model, "model.pkl")
joblib.dump(le, "label_encoder.pkl")
joblib.dump(list(X.columns), "symptom_columns.pkl")  # needed by the Django app to build the input form
